# 3.4 Neural ODE 与 Poisson：可复现数值实验

不采用随机数据；先计算参数拟合与跨积分器误差，再检查一维 Poisson 中心差分的网格误差。

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

A_TRUE, T, H = 1.6, 2.0, 0.2
TRAIN_Y0 = np.array([0.8, 1.0, 1.2])
TEST_Y0 = np.array([0.9, 1.4, 2.0])
T_TRAIN = np.linspace(0, T, 11)
T_TEST = np.linspace(0, T, 101)

def amplification(a, h, solver):
    z = -a * h
    if solver == "Euler":
        return 1 + z
    return 1 + z + z**2/2 + z**3/6 + z**4/24

def train_mse(a, solver):
    factors = amplification(a, H, solver) ** np.arange(len(T_TRAIN))
    pred = TRAIN_Y0[:, None] * factors[None, :]
    exact = TRAIN_Y0[:, None] * np.exp(-A_TRUE * T_TRAIN)[None, :]
    return np.mean((pred - exact)**2)

for solver in ["Euler", "RK4"]:
    result = minimize_scalar(
        lambda a: train_mse(a, solver),
        bounds=(0.5, 2.5), method="bounded",
        options={"xatol": 1e-14},
    )
    a_fit = result.x
    pred = np.stack([
        solve_ivp(
            lambda t, y: -a_fit*y, (0, T), [y0],
            t_eval=T_TEST, method="DOP853",
            rtol=1e-12, atol=1e-14,
        ).y[0]
        for y0 in TEST_Y0
    ])
    exact = TEST_Y0[:, None] * np.exp(-A_TRUE * T_TEST)[None, :]
    diff = pred - exact
    print(solver, "a=", a_fit,
          "train MSE=", train_mse(a_fit, solver),
          "test RMSE=", np.sqrt(np.mean(diff**2)),
          "test max error=", np.max(np.abs(diff)))
    print("Final-point errors (y0=1.4):")
    for h in [0.2, 0.1, 0.05]:
        n = round(T / h)
        yn = 1.4 * amplification(a_fit, h, solver)**n
        print(" h=", h, " error=", abs(yn-1.4*np.exp(-A_TRUE*T)))

print("Poisson finite-difference errors:")
for N in [5, 10, 20, 40]:
    dx = 1.0/N
    x = np.arange(N+1)*dx
    coefficient = np.pi**2*dx**2/(4*np.sin(np.pi*dx/2)**2)
    approximate = coefficient*np.sin(np.pi*x)
    error = np.max(np.abs(approximate-np.sin(np.pi*x)))
    print("N=", N, "dx=", dx, "max error=", error)